# Real-time form matching against reference videos

The baseline squat counter used fixed knee-angle cutoffs (`start_angle=150`, `deep_angle=95`, `stand_angle=160`). Those numbers only work for one camera setup and one exercise.

This iteration scores a **live stream** against a **phase template** built from reference clips:

1. Offline, extract joint-angle sequences from good-form videos and average them into a cyclic template indexed by movement phase.
2. Online, each new frame is compared only to a small neighborhood of the current phase. That is local template following, not batch DTW, so feedback does not wait for the repetition to finish.

Until you drop clips into `data/references/squat/good/`, the notebook bootstraps a synthetic template so the matcher can be tested without a camera.

In [ ]:
%pip install numpy opencv-python mediapipe

In [ ]:
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np

from src.visualization.overlay import draw_match, draw_skeleton
from src.pose.stream import features_from_result, make_landmarker, monotonic_timestamp_ms, to_mp_image
from src.experimental.realtime_matcher import RealtimeReferenceMatcher
from src.experimental.synthetic import synthetic_squat_sequence
from src.experimental.template import MultiViewTemplates, build_multiview_templates

## Build or load a reference template

Record a few good-form clips, then run:

`python scripts/experimental/build_template.py --videos data/references/squat/good --out data/templates/squat_good.npz`

See `data/README.md` for public workout-video datasets you can mine the same way.

In [ ]:
template_path = Path("data/templates/squat_good.npz")
if template_path.exists():
    templates = MultiViewTemplates.load(template_path)
    print(f"Loaded {template_path}")
else:
    templates = build_multiview_templates(
        {
            "side": [synthetic_squat_sequence(n_frames=72, n_reps=3, depth=88, noise=1.0, view="side")],
            "front": [synthetic_squat_sequence(n_frames=72, n_reps=3, depth=88, noise=1.0, view="front")],
        }
    )
    templates.save(template_path)
    print(f"Wrote synthetic bootstrap template to {template_path}")
print("views:", list(templates.views))

## Offline check (no camera)

A matching synthetic squat should complete reps. A shallow copy of the same motion should produce knee residuals versus the deep reference.

In [ ]:
def run_sequence(seq):
    matcher = RealtimeReferenceMatcher(templates)
    history = []
    for row in seq:
        history.append(matcher.update(row))
    return history

good = run_sequence(synthetic_squat_sequence(n_frames=128, n_reps=2, depth=90, noise=1.0))
shallow = run_sequence(synthetic_squat_sequence(n_frames=128, n_reps=2, depth=140, noise=1.0))
print("matching reps:", good[-1].reps, "last feedback:", good[-1].feedback)
print("shallow reps:", shallow[-1].reps, "last feedback:", shallow[-1].feedback)
print("shallow knee |z| at bottom:", max(abs(r.z_scores[0]) for r in shallow))

## Live webcam

Each frame: pose → joint-angle features → local phase search on the template → residual feedback. Press `q` to quit.

In [ ]:
matcher = RealtimeReferenceMatcher(templates)
cap = cv2.VideoCapture(0)
last_ts = -1
frame_index = 0

try:
    with make_landmarker() as landmarker:
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                print("Video capture ended.")
                break

            ts = monotonic_timestamp_ms(cap, last_ts, frame_index)
            last_ts = ts
            frame_index += 1

            result = landmarker.detect_for_video(to_mp_image(frame), ts)
            features, view, landmarks = features_from_result(result)
            if landmarks is not None:
                draw_skeleton(frame, landmarks)
            match = matcher.update(features, view=view)
            draw_match(frame, match)

            cv2.imshow("Reference form matching", cv2.resize(frame, (960, 540)))
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
finally:
    cap.release()
    cv2.destroyAllWindows()